<a href="https://www.kaggle.com/code/abigailbuckland/project2?scriptVersionId=368840295" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import glob

print(glob.glob('/kaggle/input/**/RA_data.csv', recursive=True))

['/kaggle/input/datasets/abigailbuckland/ambiguity/RA_data.csv']


In [2]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/abigailbuckland/ambiguity/RA_data.csv')

df.head()

,LQ_CODE,CO_1,CO_2,SY_1,SY_2,SENTENCE,JUSTIFICATION
0,AMB,Real-Time Monitoring System for IoT Device Health,Automated Diagnostic Alerts for Predictive Mai...,Device Health Monitor,NaN,The device health monitor should continuously ...,'Device Health Monitor' handles both device mo...
1,AMB,User Session Encryption Protocol for Data Secu...,Session Timeout Enforcement for Unauthorized A...,Session Security Layer,NaN,The session security layer must encrypt user s...,'Session Security Layer' covers both encryptio...
2,AMB,Automated Identity Verification for Onboarding,Biometric Authentication for Secure Access,Identity Verification System,NaN,The identity verification system should verify...,'Identity Verification System' addresses both ...
3,AMB,Predictive Sales Analytics for Market Trend Fo...,Behavioral Analysis for Consumer Insights,Analytics Engine,NaN,The analytics engine should forecast sales tre...,'Analytics Engine' is tasked with both trend f...
4,AMB,Content Filtering Mechanism for Platform Polic...,Sentiment Analysis Tool for Content Quality Ev...,Content Management System,NaN,The content management system should filter po...,'Content Management System' covers both filter...


In [3]:
print(df.columns.tolist())

['LQ_CODE', 'CO_1', 'CO_2', 'SY_1', 'SY_2', 'SENTENCE', 'JUSTIFICATION']


In [4]:

# Keep the columns needed for classification
df = df[['SENTENCE', 'LQ_CODE']].copy()

# Rename columns
df = df.rename(columns={
    'SENTENCE': 'text',
    'LQ_CODE': 'label'
})

# Remove missing text or labels
df = df.dropna(subset=['text', 'label'])

# Remove extra whitespace
df['text'] = df['text'].str.strip()
df['label'] = df['label'].str.strip()

# Remove empty text or labels
df = df[(df['text'] != '') & (df['label'] != '')]

# Remove exact duplicate rows
df = df.drop_duplicates()

# Print total number of examples
print("Total examples:", len(df))

# Print number of examples in each class
print("\nClass counts:")
print(df['label'].value_counts())

# Preview cleaned dataset
df.head()


Total examples: 5875

Class counts:
label
HQS    1424
MLN    1249
INC    1102
AMB    1086
RDC    1014
Name: count, dtype: int64


,text,label
0,The device health monitor should continuously ...,AMB
1,The session security layer must encrypt user s...,AMB
2,The identity verification system should verify...,AMB
3,The analytics engine should forecast sales tre...,AMB
4,The content management system should filter po...,AMB


In [5]:

from sklearn.model_selection import train_test_split

# Set a fixed random seed
SEED = 42

# Split into 70% training and 30% temporary data
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

# Split temporary data into 15% validation and 15% testing
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Print dataset sizes
print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Testing examples:", len(test_df))

# Print class distribution for each split
print("\nTraining class counts:")
print(train_df["label"].value_counts())

print("\nValidation class counts:")
print(val_df["label"].value_counts())

print("\nTesting class counts:")
print(test_df["label"].value_counts())


Training examples: 4112
Validation examples: 881
Testing examples: 882

Training class counts:
label
HQS    997
MLN    874
INC    771
AMB    760
RDC    710
Name: count, dtype: int64

Validation class counts:
label
HQS    213
MLN    187
INC    166
AMB    163
RDC    152
Name: count, dtype: int64

Testing class counts:
label
HQS    214
MLN    188
INC    165
AMB    163
RDC    152
Name: count, dtype: int64


In [6]:
!nvidia-smi

Thu Nov 12 15:18:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# Convert sentences into numbers
tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(train_df["text"])
X_val = tfidf.transform(val_df["text"])

# Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, train_df["label"])

# Make predictions
predictions = model.predict(X_val)

# Show results
print("Macro-F1:", f1_score(val_df["label"], predictions, average="macro"))
print(classification_report(val_df["label"], predictions))


Macro-F1: 0.9675066318684195
              precision    recall  f1-score   support

         AMB       0.98      0.99      0.98       163
         HQS       0.95      0.99      0.97       213
         INC       0.95      0.99      0.97       166
         MLN       0.97      0.91      0.94       187
         RDC       0.99      0.95      0.97       152

    accuracy                           0.97       881
   macro avg       0.97      0.97      0.97       881
weighted avg       0.97      0.97      0.97       881

